# Part 2, Notebook 3: Models and raw outputs

Notebook 2 prepared model-ready batches. Here we build two classifiers and inspect their outputs before training.

You will learn why an image batch has shape `[batch, 3, 224, 224]`, why a three-class classifier outputs `[batch, 3]`, and how logits become probabilities, IDs, and class names.

> We inspect models here. We do not train or evaluate on the official test set.

In [1]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")
print("CPU is expected on this laptop.")

PyTorch: 2.13.0+cpu
Device:  cpu
CPU is expected on this laptop.


## 1. Load Notebook 2 outputs

Notebook 2 saved crop files and metadata.csv. This notebook reads that saved bridge instead of depending on Notebook 2 still being open. We use validation images while developing; test remains the final exam.

In [2]:
CURRENT_FOLDER = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_FOLDER if (CURRENT_FOLDER / "pyproject.toml").exists() else CURRENT_FOLDER.parent
METADATA_PATH = PROJECT_ROOT / "data" / "classification_crops" / "metadata.csv"
SELECTED_CLASSES = ("Abyssinian", "Bengal", "Egyptian Mau")

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Notebook 2 outputs are missing. Run notebooks/02_classification_data.ipynb first."
    )

metadata = pd.read_csv(METADATA_PATH)
assert len(metadata) == 584
assert set(metadata["label_id"]) == {0, 1, 2}
display(metadata.groupby(["split", "class_name"]).size().unstack(fill_value=0))
print(f"Loaded {len(metadata)} crop records.")

class_name,Abyssinian,Bengal,Egyptian Mau
split,,,
test,98,100,96
train,80,80,72
val,20,20,18


Loaded 584 crop records.


## 2. Rebuild one deterministic validation batch

After a notebook restart, Python forgets variables, so we define the small Dataset class again. The crop files and metadata are the durable connection between notebooks.

Each validation crop receives the same resize, tensor conversion, and ImageNet normalization every time.

In [3]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PetCropDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, project_root: Path, transform=None):
        self.rows = rows.reset_index(drop=True).copy()
        self.project_root = project_root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        row = self.rows.iloc[index]
        with Image.open(self.project_root / row["crop_path"]) as image:
            image = image.convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
        return image, int(row["label_id"])

train_rows = metadata[metadata["split"] == "train"].copy()
val_rows = metadata[metadata["split"] == "val"].copy()
val_dataset = PetCropDataset(val_rows, PROJECT_ROOT, evaluation_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

image_batch, true_label_batch = next(iter(val_loader))
assert image_batch.shape == (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
assert true_label_batch.shape == (BATCH_SIZE,)
print(f"Validation image batch: {tuple(image_batch.shape)}")
print(f"Validation labels:      {tuple(true_label_batch.shape)}")

Validation image batch: (16, 3, 224, 224)
Validation labels:      (16,)


## 3. Establish a majority-class baseline

A baseline is a deliberately simple comparison. Here it always predicts the most common training class. If a trained model cannot beat this validation score, it has not learned something useful.

Abyssinian and Bengal tie in training. Choosing the smaller label ID breaks that tie repeatably.

In [4]:
train_class_counts = train_rows["label_id"].value_counts().sort_index()
majority_label_id = int(train_class_counts.idxmax())
majority_class_name = SELECTED_CLASSES[majority_label_id]
majority_val_accuracy = (val_rows["label_id"] == majority_label_id).mean()

display(train_class_counts.rename(index=dict(enumerate(SELECTED_CLASSES))).to_frame("training examples"))
print(f"Majority-class rule: always predict {majority_class_name} ({majority_label_id})")
print(f"Validation accuracy of that rule: {majority_val_accuracy:.1%}")
print("The official test split was not used for this baseline.")

,training examples
label_id,
Abyssinian,80
Bengal,80
Egyptian Mau,72


Majority-class rule: always predict Abyssinian (0)
Validation accuracy of that rule: 34.5%
The official test split was not used for this baseline.


## 4. Build a tiny convolutional neural network

A convolutional neural network (CNN) learns visual patterns with small filters. Early layers can notice edges and textures; deeper layers combine them into more useful patterns.

This TinyCNN begins with random weights. Its predictions are not expected to be correct yet; it teaches the complete input-to-output shape.

In [5]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images)
        flattened = torch.flatten(features, start_dim=1)
        return self.classifier(flattened)

tiny_cnn = TinyCNN(num_classes=len(SELECTED_CLASSES)).to(DEVICE)
tiny_parameter_count = sum(parameter.numel() for parameter in tiny_cnn.parameters())
print(tiny_cnn)
print(f"TinyCNN trainable parameters: {tiny_parameter_count:,}")

TinyCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Linear(in_features=32, out_features=3, bias=True)
)
TinyCNN trainable parameters: 5,187


## 5. Logits, softmax probabilities, IDs, and names

The final layer produces three raw scores per image: one score for each class. These raw scores are logits. They can be negative and do not have to add to one.

Softmax converts three logits into three probabilities. The largest probability gives the predicted label ID, which maps to a class name. Because TinyCNN is untrained, these predictions are only a format demonstration.

In [6]:
tiny_cnn.eval()
with torch.no_grad():
    tiny_logits = tiny_cnn(image_batch.to(DEVICE))
    tiny_probabilities = torch.softmax(tiny_logits, dim=1)
    tiny_predicted_ids = tiny_probabilities.argmax(dim=1)

assert tiny_logits.shape == (BATCH_SIZE, len(SELECTED_CLASSES))
assert torch.allclose(tiny_probabilities.sum(dim=1), torch.ones(BATCH_SIZE), atol=1e-6)

tiny_preview = pd.DataFrame({
    "true_class": [SELECTED_CLASSES[label_id] for label_id in true_label_batch[:5].tolist()],
    "predicted_id": tiny_predicted_ids[:5].tolist(),
    "predicted_class": [SELECTED_CLASSES[label_id] for label_id in tiny_predicted_ids[:5].tolist()],
})
for class_id, class_name in enumerate(SELECTED_CLASSES):
    tiny_preview[f"P({class_name})"] = tiny_probabilities[:5, class_id].tolist()

print(f"TinyCNN logits shape:        {tuple(tiny_logits.shape)}")
print(f"First image raw logits:      {tiny_logits[0].tolist()}")
print(f"First image probabilities:   {tiny_probabilities[0].tolist()}")
print(f"First image probability sum: {tiny_probabilities[0].sum().item():.6f}")
display(tiny_preview.round(3))

TinyCNN logits shape:        (16, 3)
First image raw logits:      [-0.04736115038394928, 0.11149754375219345, 0.08279941976070404]
First image probabilities:   [0.3020068407058716, 0.35400402545928955, 0.34398916363716125]
First image probability sum: 1.000000


,true_class,predicted_id,predicted_class,P(Abyssinian),P(Bengal),P(Egyptian Mau)
0,Abyssinian,1,Bengal,0.302,0.354,0.344
1,Abyssinian,1,Bengal,0.313,0.360,0.326
2,Abyssinian,1,Bengal,0.309,0.359,0.332
3,Abyssinian,2,Egyptian Mau,0.290,0.349,0.361
4,Abyssinian,1,Bengal,0.307,0.353,0.341


## 6. Load pretrained ResNet18 and replace its classifier

ResNet18 is a larger CNN trained on ImageNet. Its feature-extraction layers already contain useful general visual patterns: this is transfer learning.

The old final layer predicted 1,000 ImageNet classes. We replace it with a new three-output layer for our breeds. This new head starts random, so it is not ready to judge until training.

The first run may download official pretrained weights once. Later runs reuse the cached copy.

In [7]:
try:
    resnet_weights = models.ResNet18_Weights.DEFAULT
    resnet18 = models.resnet18(weights=resnet_weights)
except Exception as error:
    raise RuntimeError(
        "Could not load pretrained ResNet18 weights. Check Internet for the first download, then retry."
    ) from error

old_feature_count = resnet18.fc.in_features
resnet18.fc = nn.Linear(old_feature_count, len(SELECTED_CLASSES))
resnet18 = resnet18.to(DEVICE)
resnet_parameter_count = sum(parameter.numel() for parameter in resnet18.parameters())

print(f"Old ResNet feature count: {old_feature_count}")
print(f"New final layer:          {resnet18.fc}")
print(f"ResNet18 parameters:      {resnet_parameter_count:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [03:30<00:00, 223kB/s] 


Old ResNet feature count: 512
New final layer:          Linear(in_features=512, out_features=3, bias=True)
ResNet18 parameters:      11,178,051


In [8]:
resnet18.eval()
with torch.no_grad():
    resnet_logits = resnet18(image_batch.to(DEVICE))
    resnet_probabilities = torch.softmax(resnet_logits, dim=1)
    resnet_predicted_ids = resnet_probabilities.argmax(dim=1)

assert resnet_logits.shape == (BATCH_SIZE, len(SELECTED_CLASSES))
assert torch.allclose(resnet_probabilities.sum(dim=1), torch.ones(BATCH_SIZE), atol=1e-6)

resnet_preview = pd.DataFrame({
    "true_class": [SELECTED_CLASSES[label_id] for label_id in true_label_batch[:5].tolist()],
    "predicted_class": [SELECTED_CLASSES[label_id] for label_id in resnet_predicted_ids[:5].tolist()],
})
for class_id, class_name in enumerate(SELECTED_CLASSES):
    resnet_preview[f"P({class_name})"] = resnet_probabilities[:5, class_id].tolist()

print(f"ResNet18 logits shape:      {tuple(resnet_logits.shape)}")
print(f"First image raw logits:     {resnet_logits[0].tolist()}")
print(f"First image probabilities:  {resnet_probabilities[0].tolist()}")
print(f"First image probability sum:{resnet_probabilities[0].sum().item():.6f}")
display(resnet_preview.round(3))
print("These are not performance results: the new three-class head is untrained.")

ResNet18 logits shape:      (16, 3)
First image raw logits:     [0.5498502254486084, 0.01010245829820633, -0.2176457941532135]
First image probabilities:  [0.4885033071041107, 0.2847462594509125, 0.22675049304962158]
First image probability sum:1.000000


,true_class,predicted_class,P(Abyssinian),P(Bengal),P(Egyptian Mau)
0,Abyssinian,Abyssinian,0.489,0.285,0.227
1,Abyssinian,Abyssinian,0.529,0.297,0.174
2,Abyssinian,Bengal,0.278,0.463,0.259
3,Abyssinian,Bengal,0.241,0.441,0.318
4,Abyssinian,Bengal,0.369,0.374,0.257


These are not performance results: the new three-class head is untrained.


## Checkpoint before Notebook 4

The classification path is:

`[16, 3, 224, 224]` image batch → CNN or ResNet18 → `[16, 3]` logits → softmax → predicted IDs → class names

Answer these before training:

1. Why can logits be negative while softmax probabilities cannot?
2. Why do three classes produce three logits per image?
3. Why should we not expect correct predictions from the new three-class ResNet head yet?
4. What makes ResNet18 transfer learning rather than training completely from scratch?
5. Why is a majority-class baseline useful?

Notebook 4 will teach the model: loss, gradients, backpropagation, optimizer steps, and validation metrics.